# 🚀 KAGGLE UNIVERSAL CLOUD RUNNER (32 GB VRAM DUAL T4 x2)
### Ambiente Autônomo de Alta Performance e Zero Dependências Locais
- **Hardware:** 2x NVIDIA Tesla T4 (32 GB VRAM Total em FP16)
- **Acesso Externo:** Túnel Cloudflare HTTPS Público com Certificado SSL Automático
- **Anti-Timeout Heartbeat:** Impede que o Kaggle encerre a sessão por inatividade
- **Gateway de Aplicações:** Pronto para rodar DarkLab, APIs FastAPI, LLMs e IA Generativa

In [ ]:
# =============================================================================
# PASSO 1: AUDITORIA DE HARDWARE (DUAL T4 x2 - 32GB VRAM)
# =============================================================================
!nvidia-smi
import torch

print(f"CUDA Disponível: {torch.cuda.is_available()}")
gpu_count = torch.cuda.device_count()
print(f"Total de GPUs Detectadas: {gpu_count}")
for i in range(gpu_count):
    props = torch.cuda.get_device_properties(i)
    print(f" -> GPU [{i}]: {props.name} | VRAM: {props.total_memory / (1024**3):.2f} GB")

In [ ]:
# =============================================================================
# PASSO 2: PROVISIONAMENTO DE PACOTES ESSENCIAIS & DEPENDÊNCIAS DO SISTEMA
# =============================================================================
!pip install -q --upgrade pip
!pip install -q torch accelerate transformers huggingface_hub fastapi uvicorn requests python-multipart pydantic psutil
!apt-get update -qq && apt-get install -y -qq ffmpeg aria2 curl > /dev/null 2>&1
print("✅ Pacotes e ferramentas do sistema instalados com sucesso!")

In [ ]:
# =============================================================================
# PASSO 3: SINCRONIZAÇÃO DO PROJETO COM O REPOSITÓRIO OFICIAL
# =============================================================================
import os, shutil

REPO_URL = "https://github.com/joedsondasilva199-glitch/meu-projeto.git"
TARGET_DIR = "/kaggle/working/meu-projeto"

if os.path.exists(TARGET_DIR):
    print("Atualizando repositório existente...")
    !cd {TARGET_DIR} && git pull
else:
    print("Clonando repositório oficial...")
    !git clone {REPO_URL} {TARGET_DIR}

%cd {TARGET_DIR}
print("Ambiente de trabalho posicionado em:", os.getcwd())

In [ ]:
# =============================================================================
# PASSO 4: SERVIDOR DE CONTROLE FASTAPI + EXPOSIÇÃO REMOTA CLOUDFLARE
# =============================================================================
import subprocess, threading, time, json
from fastapi import FastAPI, Body
from fastapi.middleware.cors import CORSMiddleware
import uvicorn

app = FastAPI(title="Kaggle 32GB Cloud Node", version="1.0.0")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

@app.get("/")
def status():
    return {
        "status": "online",
        "node": "Kaggle Dual T4 x2",
        "gpus": torch.cuda.device_count(),
        "cuda_device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
    }

@app.post("/exec")
def execute_command(cmd: str = Body(..., embed=True)):
    """Permite ao Antigravity ou usuário executar comandos remotamente no Kaggle via HTTPS"""
    res = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return {"cmd": cmd, "stdout": res.stdout, "stderr": res.stderr, "exit_code": res.returncode}

# 1. Inicia API FastAPI em thread background
threading.Thread(target=lambda: uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning"), daemon=True).start()
time.sleep(2)

# 2. Baixa e Inicia o Cloudflared
if not os.path.exists("/kaggle/working/cloudflared"):
    !curl -sL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /kaggle/working/cloudflared
    !chmod +x /kaggle/working/cloudflared

def start_tunnel():
    proc = subprocess.Popen(
        ["/kaggle/working/cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    for line in iter(proc.stdout.readline, ""):
        if "trycloudflare.com" in line:
            for tok in line.split():
                if "trycloudflare.com" in tok:
                    print(f"\n" + "="*60)
                    print(f"🌐 SEU NÓ KAGGLE ESTÁ ONLINE NA INTERNET:")
                    print(f"🔗 URL PÚBLICA HTTPS: {tok.strip()}")
                    print(f"🔗 SWAGGER INTERATIVO: {tok.strip()}/docs")
                    print("="*60 + "\n")
                    break

threading.Thread(target=start_tunnel, daemon=True).start()
time.sleep(5)

In [ ]:
# =============================================================================
# PASSO 5: EXECUÇÃO DO PIPELINE DE 32GB VRAM E KEEP-ALIVE CONTÍNUO
# =============================================================================
!python kaggle_32gb_runner.py

print("\n🔥 Servidor ativo e mantendo o notebook acordado. Para parar, aperte o botão 'Stop' no Kaggle.")
try:
    while True:
        time.sleep(30)
except KeyboardInterrupt:
    print("Sessão finalizada pelo usuário.")